In [ ]:
def clip_outliers_iqr(df):
    df_clipped = df.copy()
    for col in df_clipped.select_dtypes(include=['int64', 'float64']):
        Q1 = df_clipped[col].quantile(0.25)
        Q3 = df_clipped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_clipped[col] = df_clipped[col].clip(lower=lower, upper=upper)
    return df_clipped

df = clip_outliers_iqr(df)

## 3. Data Cleaning

We clean the dataset to ensure it is ready for machine learning:

- Drop or encode ID fields
- Replace placeholder missing values (`'?'`) with `NaN`
- Encode or drop low-utility columns
- Check and handle missing values
- Prepare categorical columns for encoding

In [ ]:
df.drop(['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult'], axis=1, inplace=True)


In [ ]:
# Check for placeholder '?' values
(df == '?').sum().sort_values(ascending=False)

In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
# Impute 'race' with the most frequent category
df['race'] = df['race'].fillna(df['race'].mode()[0])

# Impute diag_1/2/3 with 'Unknown'
df[['diag_1', 'diag_2', 'diag_3']] = df[['diag_1', 'diag_2', 'diag_3']].fillna('Unknown')

In [ ]:
# Actual missing values (NaNs)
df.isna().sum().sort_values(ascending=False)

In [ ]:
# 1. One-hot encode 'gender' and assign to X
df = pd.get_dummies(df, columns=['gender'])

# 2. Ensure all expected dummy columns exist (safe even if a value is missing)
for col in ['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']:
    if col not in df.columns:
        df[col] = 0

# 3. Cast the dummy columns to int
df[['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']] = df[['gender_Female', 'gender_Male', 'gender_Unknown/Invalid']].astype(int)
df.info()

In [ ]:
df = pd.get_dummies(df, columns=['race'], prefix='race')

In [ ]:
age_map = {
    '[0-10)'   : 0,
    '[10-20)'  : 1,
    '[20-30)'  : 2,
    '[30-40)'  : 3,
    '[40-50)'  : 4,
    '[50-60)'  : 5,
    '[60-70)'  : 6,
    '[70-80)'  : 7,
    '[80-90)'  : 8,
    '[90-100)' : 9
}

df['age'] = df['age'].map(age_map)

In [ ]:
drug_columns = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide',
    'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

for col in drug_columns:
    df[col] = df[col].apply(lambda x: 0 if x == 'No' else 1)

In [ ]:
df['change'] = df['change'].apply(lambda x: 1 if x == 'Ch' else 0)
df['diabetesMed'] = df['diabetesMed'].apply(lambda x: 1 if x == 'Yes' else 0)

In [ ]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].fillna('Unknown')
    df[col] = df[col].apply(map_diagnosis).astype(int)

In [ ]:
# Convert 'readmitted' to 0, 1, 2:
# 1 if '<30', 2 if '>30', 0 if 'NO'
df['readmitted_binary'] = df['readmitted'].apply(
    lambda x: 1 if x == '<30' else (2 if x == '>30' else 0)
)

# Optionally drop the original column
df.drop(columns=['readmitted'], inplace=True)

# Check class distribution
print(df['readmitted_binary'].value_counts())


In [ ]:
sample_weights = y_train.map({0: 0.618, 1: 2.987, 2: 0.954})
model.fit(X_train, y_train, sample_weight=sample_weights)


In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
        'random_state': 42
    }

    model = XGBClassifier(**params)

    # Optional: use sample weights
    sample_weights = y_train.map({0: 0.618, 1: 2.987, 2: 0.954})

    model.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, average='macro')


In [ ]:
# Split into smaller training and validation for faster tuning
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
print("Best trial:")
print(f"  F1-macro: {study.best_value:.4f}")
print("  Params:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")


In [ ]:
best_params = {
    'n_estimators': 260,
    'learning_rate': 0.02034,
    'max_depth': 10,
    'min_child_weight': 3,
    'gamma': 0.33,
    'subsample': 0.5899,
    'colsample_bytree': 0.5294,
    'reg_alpha': 0.3815,
    'reg_lambda': 1.5034,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'random_state': 42
}

In [ ]:
model = XGBClassifier(**best_params)

# Use class sample weights again
sample_weights = y_train.map({0: 0.618, 1: 2.987, 2: 0.954})

model.fit(X_train, y_train, sample_weight=sample_weights)

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))